In [ ]:
'''
import datetime
import random
import schedule
import time

# 1. Initialize the loop counter starting at 0
counter = 0


def job():
    global counter  # Reference the global counter variable
    counter += 1  # Increment the counter by 1 each run

    # Display the current time and the current run count
    current_time = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{current_time}] Hello (Run #{counter})")

    # 4. Condition: Stop the scheduler immediately once it reaches 5 runs
    if counter >= 5:
        print(">> Reached 5 runs. Canceling this scheduled job...")
        schedule.cancel_job(j)  # Cancel the specific job stored in variable 'j'


# 2. Schedule a random interval between 1 and 5 seconds, storing the job object in 'j'
j = schedule.every(1).to(5).seconds.do(job)

print("Starting the random scheduler (Max 5 runs)...")

# 3. Infinite loop to keep the script running and check the schedule
while True:
    schedule.run_pending()
    time.sleep(1)

    # Bonus: If there are no more jobs left in the scheduler, break the loop to exit the program
    if not schedule.jobs:
        print("No remaining jobs. Exiting program.")
        break
'''

In [ ]:
import datetime
import random
import schedule
import time

# ==============================================================================
# DATA ENGINEERING CONTEXT:
# สคริปต์นี้สุ่มช่วงเวลา (Random Interval Pipeline) เพื่อดึงข้อมูลหรือส่งสัญญาณแบบจำกัดจำนวนครั้ง (5 ครั้ง)
# มักใช้ในการทำ Mocking Data Stream หรือดึงข้อมูลจาก API ที่จำกัด Rate Limit (จำกัดการเรียกใช้)
# ==============================================================================

# 1. กำหนดตัวแปร Global Counter เพื่อใช้นับรอบการทำงาน (Inbound Data / Run Tracking)
counter = 0


def job():
    global counter  # อ้างอิงรอบการทำงานจากตัวแปรภายนอกฟังก์ชัน
    counter += 1  # อัปเดตจำนวนรอบเพิ่มทีละ 1 เมื่อมีข้อมูลไหลเข้ามาหรือฟังก์ชันทำงาน

    # บันทึกเวลาปัจจุบัน (Timestamp) และแสดงสถานะ Log ของการทำงาน
    # (ในงานจริง มักเปลี่ยนจากการ print เป็นการบันทึก Log ลงระบบ เช่น CloudWatch หรือดึงข้อมูลจาก API จริง)
    current_time = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{current_time}] Hello (Run #{counter})")

    # 4. ตรวจสอบเงื่อนไขการหยุดทำงาน (Exit Condition) เพื่อป้องกัน Resource รั่วไหล
    # เมื่อทำงานครบ 5 ครั้ง ให้ทำการยกเลิก Task นี้ออกจากคิวงานของ Scheduler
    if counter >= 5:
        print(">> Reached 5 runs. Canceling this scheduled job...")
        schedule.cancel_job(j)  # ลบ Job 'j' ออกจากคิวระบบของไลบรารี schedule


# 2. ตั้งเวลาการทำงานแบบ Dynamic (Random Interval) ระหว่าง 1 ถึง 5 วินาที
# ในมุม Data Engineer: การสุ่มเวลา (Random Delay) ช่วยลดโอกาสการโดนบล็อกเมื่อทำ Web Scraping
# และช่วยกระจายโหลดการทำงานไม่ให้ยิงคิวรี่ไปที่ระบบต้นทาง (Source System) ถี่เกินไป
j = schedule.every(1).to(5).seconds.do(job)

print("Starting the random scheduler (Max 5 runs)...")

# 3. ลูปหลัก (Main Engine Loop) คอยตรวจสอบและสั่งรันงานเมื่อถึงเวลาที่กำหนด
while True:
    schedule.run_pending()  # ตรวจสอบว่าในคิวมี Job ไหนถึงเวลาต้องทำงานแล้วหรือยัง
    time.sleep(1)  # พักการทำงาน 1 วินาที เพื่อประหยัดการใช้งาน CPU (CPU Resources Optimization)

    # เงื่อนไขตัดการทำงานถาวร (Graceful Shutdown)
    # หากไม่มี Job เหลืออยู่ในคิวแล้ว (โดน cancel ไปหมดแล้ว) ให้หลุดออกจาก Infinite Loop เพื่อปิดโปรแกรม
    if not schedule.jobs:
        print("No remaining jobs. Exiting program.")
        break
